In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestCentroid
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import sklearn.metrics as metrics

DATA_PATH = csv_path = os.path.join("..", "data", "EMG-data.csv")

df = pd.read_csv(DATA_PATH)
print(df['subject'].value_counts())

SUBJECT_ROWS = [99980, 103636, 111012, 105175, 100859, 99670, 102134, 100225]
TOTAL_ROWS = sum(SUBJECT_ROWS)
CHANNELS = ["channel1", "channel2", "channel3", "channel4"]
SUBJECT_SEVEN_START = sum(SUBJECT_ROWS[0:6])
LAST_TWO_NUM_ROWS = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]



subject
3    111012
4    105175
2    103636
7    102134
5    100859
8    100225
1     99980
6     99670
Name: count, dtype: int64


In [81]:

def get_feature_df(csv_path=DATA_PATH, read_channel='channel3', start_row=0, num_read=SUBJECT_ROWS[0]):
    """Given a path to EMG data csv, returns feature windowed data from specified range, exlcuding specified channels
    omits windows in which all rows do not have the same class"""
    df = pd.read_csv(csv_path)
    df = df.iloc[start_row : start_row + num_read].copy()
    dropped_channels = []
    for i in range(len(CHANNELS)):
        if read_channel != CHANNELS[i]:
            dropped_channels.append(CHANNELS[i])
    df = df.drop(columns=dropped_channels)

    windowed_class = df["class"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_class = windowed_class 
    windowed_subject = df["subject"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_rms_col = df[read_channel].pow(2).rolling(window=200, step=100).mean().pow(0.5)
    windowed_wfl_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(np.diff(x)).sum())
    windowed_stdev_col = df[read_channel].rolling(window=200, step=100).std()
    windowed_mav_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).mean())
    windowed_min_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).min())
    def crossings(s): 
        return (s.shift(1) * s < 0)
    windowed_zc_col = crossings(df[read_channel]).rolling(window=200, step=100).sum()
    windowed_max_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).max())
    
    rolling_class = df["class"].rolling(window=200, step=100)
    windowed_mask_col = rolling_class.min() == rolling_class.max()
    feature_df = pd.DataFrame({"RMS": windowed_rms_col,
                               "waveform_len": windowed_wfl_col,
                               "MAV": windowed_mav_col,
                               "max_abs": windowed_max_col,
                               "min_abs": windowed_min_col,
                               "std": windowed_stdev_col,
                               "zero_crossings": windowed_zc_col,
                               'mask': windowed_mask_col,
                               "class": windowed_class,
                               'subject': windowed_subject})
    feature_df = feature_df[feature_df['mask']]
    feature_df = feature_df[feature_df['class'].between(0, 4)]
    return feature_df

def get_train_test_df(csv_path=DATA_PATH, read_channel='channel3', train_start=0, train_read=sum(SUBJECT_ROWS[0:5]),
                  test_start=SUBJECT_SEVEN_START, test_read=LAST_TWO_NUM_ROWS):
    if (train_start + train_read > TOTAL_ROWS):
        raise ValueError('Requested train rows read out of bounds')
    if (test_start + test_read > TOTAL_ROWS):
        raise ValueError('Requested test rows read out of bounds')
    if (train_start > test_start and train_start < test_start + test_read):
        raise ValueError('Leakage in train and test sets')
    if (train_start + train_read > test_start and train_start + train_read < test_start + test_read):
            raise ValueError('Leakage in train and test sets')

    train_df = get_feature_df(csv_path,read_channel=read_channel, start_row=train_start, num_read=train_read)
    test_df = get_feature_df(csv_path, read_channel=read_channel, start_row=test_start, num_read=test_read)
    return train_df, test_df


feature_cols = ["RMS", "waveform_len", "MAV", "max_abs", "min_abs", "std", "zero_crossings"]

def get_df_features_labels(df, feature_cols=feature_cols):
    features = df[feature_cols].to_numpy()
    labels = df["class"].to_numpy()
    return features, labels

def train_nearest_centroid(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    nc = NearestCentroid()
    nc.fit(features, labels)
    return nc

def evaluate_model(model, df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    preds = model.predict(features)
    score = model.score(features, labels)
    return preds, score

def train_log_reg(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    reg = LogisticRegression(max_iter=10000)
    reg.fit(features, labels)
    return reg

def train_lda(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    lda = LinearDiscriminantAnalysis()
    lda.fit(features, labels)
    return lda

In [87]:
def get_feature_df_mc(csv_path=DATA_PATH, start_row=0, num_read=SUBJECT_ROWS[0]):
    mc_df = None
    for channel in CHANNELS:
        df_ch = get_feature_df(csv_path=csv_path, read_channel=channel, start_row=start_row, num_read=num_read)
        channel_features = df_ch[feature_cols].add_suffix(f"_{channel}")
        if mc_df is None:
            mc_df = channel_features
            mc_df["class"] = df_ch["class"]
            mc_df["subject"] = df_ch["subject"]
        else:
            mc_df = pd.concat([mc_df, channel_features], axis=1)
    return mc_df
feature_cols_mc = [f"{col}_{channel}" for channel in CHANNELS for col in feature_cols]

In [88]:
def evaluate_model_detailed(model, df, feature_cols=feature_cols, f1_average='macro'):
    preds, score = evaluate_model(model, df, feature_cols=feature_cols)
    labels = df['class'].to_numpy()

    f1 = metrics.f1_score(labels, preds, average=f1_average)
    bacc = metrics.balanced_accuracy_score(labels, preds)
    confusion_matrix = metrics.confusion_matrix(labels, preds)

    results = {
        'preds': preds,
        'score': score,
        'f1': f1,
        'bacc': bacc,
        'confusion matrix': confusion_matrix
    }

    return results
    

In [84]:
train_df, test_df = get_train_test_df()
reg = train_log_reg(train_df)

train_results = evaluate_model_detailed(reg, train_df)
test_results = evaluate_model_detailed(reg, test_df)

print('Single channel: ')
print(f'Train results:')
print(f"score: {train_results['score']}")
print(f"f1: {train_results['f1']}")
print(f"bacc: {train_results['bacc']}")
print(f"confusion matrix: {train_results['confusion matrix']}")

print(f'Test results: ')
print(f"score: {test_results['score']}")
print(f"f1: {test_results['f1']}")
print(f"bacc: {test_results['bacc']}")
print(f"confusion matrix: {test_results['confusion matrix']}")

Single channel: 
Train results:
score: 0.7275547799108009
f1: 0.7263311660908822
bacc: 0.7278364280661246
confusion matrix: [[1024    0    0    0    0]
 [   0  782    5  216   16]
 [   1    0  695   15  305]
 [   3  127   33  802  100]
 [   0    0  443  141  449]]
Test results: 
score: 0.7210578842315369
f1: 0.7109644194071109
bacc: 0.7218597427899374
confusion matrix: [[146   0 256   0   0]
 [  0 321   7  79   7]
 [  0   0 364   0  34]
 [  0  52   0 338   1]
 [  0   0   0 123 276]]


In [86]:
train_df_mc = get_feature_df_mc(start_row=0, num_read=sum(SUBJECT_ROWS[0:6]))
test_df_mc = get_feature_df_mc(start_row=SUBJECT_SEVEN_START, num_read=LAST_TWO_NUM_ROWS)

reg_mc = train_log_reg(train_df_mc, feature_cols=feature_cols_mc)

train_results_mc = evaluate_model_detailed(reg_mc, train_df_mc, feature_cols=feature_cols_mc)
test_results_mc = evaluate_model_detailed(reg_mc, test_df_mc, feature_cols=feature_cols_mc)

print('Multi-channel:')
print('Train results:')
print(f"score: {train_results_mc['score']}")
print(f"f1: {train_results_mc['f1']}")
print(f"bacc: {train_results_mc['bacc']}")
print(f"confusion matrix:\n{train_results_mc['confusion matrix']}")

print('Test results:')
print(f"score: {test_results_mc['score']}")
print(f"f1: {test_results_mc['f1']}")
print(f"bacc: {test_results_mc['bacc']}")
print(f"confusion matrix:\n{test_results_mc['confusion matrix']}")

Multi-channel:
Train results:
score: 0.85546875
f1: 0.8568945948180499
bacc: 0.8565677494189059
confusion matrix:
[[1221    0    0    0    0]
 [   0  966    7  223   21]
 [   1    0 1110   24   76]
 [   6  156   46  884  171]
 [   0   11    8  138 1075]]
Test results:
score: 0.8068862275449101
f1: 0.8071102418694508
bacc: 0.8057626335536249
confusion matrix:
[[369   0  33   0   0]
 [  0 383   6   6  19]
 [  0   0 247   0 151]
 [  0  56   0 318  17]
 [  0   5   0  94 300]]
